In [10]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

LOADING DATA

In [3]:
fake_data = pd.read_csv('fake_news_data.csv')

TEXT PREPROCESSING

In [4]:
y = fake_data['fake_or_factual']
lemmatize = WordNetLemmatizer()

fake_data['lowercased'] = fake_data['text'].apply(lambda x: x.lower())
fake_data['no_punct'] = fake_data['lowercased'].apply(lambda x: re.sub(r"[^\s\w]",'', x))
fake_data['no_punct'] = fake_data['lowercased'].apply(lambda x: re.sub(r"[\d]",'', x))
fake_data['tokenized'] = fake_data['no_punct'].apply(lambda x : word_tokenize(x))
fake_data['lemmatized'] = fake_data['tokenized'].apply(lambda x : [lemmatize.lemmatize(token) for token in x])
final_data = fake_data['lemmatized'].apply(lambda x: " ".join(x))

VECTORIZING TEXT

In [5]:
bow = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 2)
)
bow_doc = bow.fit_transform(final_data)
bow_array = bow_doc.toarray()
bow_df = pd.DataFrame(bow_array, columns=bow.get_feature_names_out())
bow_df.tail(20)

,aapl,aapl iphones,aaron,aaron bernstein,aarp,aarp ad,aarp youtube,abadi,abadi commander,abadi declared,...,zone celebrated,zone listen,zone upcoming,zoom,zoom charity,zouka,zouka killed,zuckerberg,zuckerberg ha,zuckerberg hint
178,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
179,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
180,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
181,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
182,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
183,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
184,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
185,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
186,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
187,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


VECTORIZING TEXT USING TFIDF

In [46]:
tfidf = TfidfVectorizer(
    stop_words='english',
)
tfidf_doc = tfidf.fit_transform(final_data)
tfidf_array = tfidf_doc.toarray()
tfidf_df = pd.DataFrame(tfidf_array, columns=tfidf.get_feature_names_out())
tfidf_df
print(len(tfidf.get_feature_names_out()))

7887


SPLIT DATA INTO TRAINING AND TESTING SET

In [47]:
x_train, x_test, y_train, y_test = train_test_split(tfidf_df, y, test_size=0.3, random_state=7)  

TRAIN LOGISTIC REGRESSION MODEL

In [48]:
lr = LogisticRegression(random_state=7)
model_training = lr.fit(x_train, y_train)
y_predict_class = lr.predict(x_test)
print(classification_report(y_test, y_predict_class))

              precision    recall  f1-score   support

Factual News       0.82      0.79      0.81        29
   Fake News       0.81      0.84      0.83        31

    accuracy                           0.82        60
   macro avg       0.82      0.82      0.82        60
weighted avg       0.82      0.82      0.82        60



TRAIN NAIVE BAYES MODEL

In [49]:
nb = MultinomialNB()
nb.fit(x_train, y_train)
y_predict_class_2 = nb.predict(x_test)
print(classification_report(y_test, y_predict_class_2))

              precision    recall  f1-score   support

Factual News       0.75      0.83      0.79        29
   Fake News       0.82      0.74      0.78        31

    accuracy                           0.78        60
   macro avg       0.79      0.78      0.78        60
weighted avg       0.79      0.78      0.78        60



CLASSIFICATION REPORT

In [82]:
print(classification_report(y_test, y_predict_class))

              precision    recall  f1-score   support

Factual News       0.83      0.83      0.83        29
   Fake News       0.84      0.84      0.84        31

    accuracy                           0.83        60
   macro avg       0.83      0.83      0.83        60
weighted avg       0.83      0.83      0.83        60



CONFUSION MATRIX

In [83]:
print(confusion_matrix(y_test, y_predict_class))

[[24  5]
 [ 5 26]]
